In [ ]:
# VLA : Vision Language Model
# 멀티모달

# YOLO : 정보억제, 경량화
# image -> YOLO -> text -> LLM -> response

In [1]:
# PinkWink/pinky_mujoco_menagerie/pinky_mujoco_menagerie/scripts/best.pt
!pip install ultralytics

In [3]:
import cv2
from ultralytics import YOLO
from PIL import Image

# 이미지 테스트
img_path = "./mujoco_objects.png"
frame = cv2.resize(cv2.imread(img_path), (640, 480))
model = YOLO("best.pt")
results = model(source=frame, conf=0.5, verbose=True)

/home/user/.pyenv/versions/3.12.10/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(



0: 480x640 1 can, 1 milk, 1 lemon, 13.9ms
Speed: 2.2ms preprocess, 13.9ms inference, 18.3ms postprocess per image at shape (1, 3, 480, 640)


In [4]:
results

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'can', 1: 'milk', 2: 'lemon'}
 obb: None
 orig_img: array([[[ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17],
         ...,
         [ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17]],
 
        [[ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17],
         ...,
         [ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17]],
 
        [[ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17],
         ...,
         [ 51,  34,  17],
         [ 51,  34,  17],
         [ 51,  34,  17]],
 
        ...,
 
        [[ 90,  61,  35],
         [ 85,  57,  28],
         [ 85,  57,  28],
         ...,
         [142, 114,  85],
         [139, 111,  84],
         [125,  90,  78]],
 
        [[104,  86,  68],
         [ 90,  61,  35],
         [ 85,  57,  28],
         ...

In [5]:
for i, r in enumerate(results):
    im_bgr = r.plot()
    im_rgb = Image.fromarray(im_bgr[..., ::-1])
    r.show()

In [8]:
# 실시간 테스트
import numpy
import mujoco as mj
from mujoco.glfw import glfw        # 시뮬레이터 (OpenGL기반)

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))

from utils.mouse_callbacks import MouseCallbacks            # 마우스 이벤트 (카메라 회전/이동/줌 등) 콜백
from utils.keyboard_callbacks import KeyboardCallbacks     # 키보드 입력(로봇 제어 및 초기화) 콜백
from utils.scene_creator import SceneCreator        # MJCF xml 파일 조합 및 scene 빌드용 핼퍼

In [9]:
cwd = os.getcwd()
PROJECT_ROOT = os.path.dirname(cwd)
print(f"project root: {PROJECT_ROOT}")

assets_dir = os.path.join(PROJECT_ROOT, "assets")
base_env_path = os.path.join(assets_dir, "scenes", "floor_sky.xml")     # 환경 MJCF
robot_path = os.path.join(assets_dir, "robots", "pinky", "pinky.xml")   # 로봇 MJCF

# 배치할 오브젝트 정보
objects_to_spawn = [
    {"path": os.path.join(assets_dir, "objects", "can.xml"), "name":"can_1", "pos": "0.5 0.3 0.05"},
    {"path": os.path.join(assets_dir, "objects", "milk.xml"), "name": "milk_1", "pos":"0.5 0.0 0.05"},
    {"path": os.path.join(assets_dir, "objects", "lemon.xml"), "name": "lemon_1", "pos":"0.5 -0.3 0.05"}
]

# 각 MJCF들을 병합
scene_xml_string = SceneCreator.build_mjcf_scene(
    base_env_path = base_env_path,
    robot_path = robot_path,
    objects_to_spawn=objects_to_spawn,
    save_xml=False
)

project root: /home/user/source/python312/python


AttributeError: type object 'SceneCreator' has no attribute 'build_mjcf_scene'

In [ ]:
# XML문자열로부터 MuJoCo 모델(물리/구조 정보) 생성
model = mj.MjModel.from_xml_string(scene_xml_string)
# 모델에 대한 시뮬레이션 데이터(상태 변수, 센서값 등) 생성
data = mj.MjData(model)

# mujoco 카메라(메인, 로봇 시점), 옵션, 씬(각 시점의 3D 그래픽 정보) 객체 선언
main_cam = mj.MjvCamera()
robot_cam = mj.MjvCamera()
opt = mj.MjvOption()
scene_main = mj.MjvScene(model, maxgeom=10000)
scene_robot = mj.MjvScene(model, maxgeom=10000)

In [ ]:
# GLFW 초기화 및 윈도우 생성
glfw.init()
window_main = glfw.create_window(900, 900, "Main View", None, None)
window_robot = glfw.create_window(640, 480, "Camera View", None, window_main)
glfw.hide_window(window_robot)

# OpenGL 컨텍스트 연결 및 렌더 컨텍스트 생성
glfw.make_context_current(window_main)
ctx_main = mj.MjrContext(model, mj.mjtFontScale.mjFONTSCALE_150.value)
glfw.make_context_current(window_robot)
ctx_robot = mj.MjrContext(model, mj.mjtFontScale.mjFONTSCALE_150.value)

# 수직 동기화 설정 : [0]비활성화 [1]활성화
glfw.swap_interval(1)
# Free Camera, 렌더링 옵션 기본값
mj.mjv_defaultFreeCamera(model, main_cam)
mj.mjv_defaultOption(opt)

In [ ]:
main_cam.type = mj.mjtCamera.mjCAMERA_FREE
main_cam.azimuth = 0        # 카메라 수평(좌우) 회전 각도
main_cam.elevation = -30    # 카메라 수직(상하) 회전 각도
main_cam.distance = 2       # 카메라와 대상(lookat) 사이 거리
main_cam.lookat = np.array([0.0, 0.0, 0.5])     # 카메라가 바라보는 좌표

# 로봇 카메라 : 고정(Fixed) 모드 및 특정 카메라 ID 로 지정
robot_cam.type = mj.mjtCamera.mjCAMERA_FIXED
cam_id = mj.mj_name2id(model, mj.mjtObj.mjOBJ_CAMERA, "camera")
robot_cam.fixedcamid = cam_id

# 메인 씬 : 그림자/반사 효과 비활성화 (성능 향상 목적)
scene_main.flags[mj.mjtRndFlag.mjRND_SHADOW] = False
scena_main.flags[mj.mjtRndFlag.mjRND_REFLECTION] = False